# Customer Churn Prediction & Retention Dashboard

End-to-end pipeline: synthetic data generation → EDA → feature engineering →
handling class imbalance → model comparison → evaluation → SHAP explainability →
saving the winning model (for the Streamlit dashboard).

**Note:** No real dataset was attached to this project, so this notebook generates a
realistic **synthetic** telecom dataset (7,000 customers) shaped like the classic
"Telco Customer Churn" dataset. To use your own data, just replace the
`generate_synthetic_data()` call in cell 2 with `pd.read_csv("your_file.csv")`.

## 1. Imports

In [ ]:
import numpy as np                                   # numeric arrays & random data generation
import pandas as pd                                   # dataframes for tabular data
import joblib                                          # saving/loading trained models to disk

from sklearn.model_selection import train_test_split   # splits data into train/test sets
from sklearn.preprocessing import LabelEncoder          # converts text categories -> numbers
from sklearn.linear_model import LogisticRegression     # baseline model #1
from sklearn.ensemble import RandomForestClassifier     # baseline model #2
from xgboost import XGBClassifier                       # the "winning" model
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

from imblearn.over_sampling import SMOTE                # oversamples the minority (churned) class

import shap                                             # explains WHY the model predicts churn
import matplotlib.pyplot as plt

shap.initjs()  # enables nice interactive SHAP plots in the notebook

## 2. Generate synthetic data

Replace this cell with `df = pd.read_csv("your_file.csv")` once you have a real dataset.

In [ ]:
def generate_synthetic_data(n=7000, seed=42):
    rng = np.random.default_rng(seed)                   # reproducible random generator

    tenure = rng.integers(0, 72, n)                      # months as a customer (0-72)
    monthly_charges = rng.uniform(18, 120, n).round(2)   # $ billed per month
    contract = rng.choice(
        ["Month-to-month", "One year", "Two year"], n, p=[0.55, 0.25, 0.20]
    )
    internet_service = rng.choice(["DSL", "Fiber optic", "No"], n, p=[0.35, 0.45, 0.20])
    tech_support = rng.choice(["Yes", "No"], n, p=[0.4, 0.6])
    payment_method = rng.choice(
        ["Electronic check", "Mailed check", "Bank transfer", "Credit card"], n
    )
    total_charges = (monthly_charges * (tenure + 1)).round(2)  # roughly cumulative billing

    # Build churn probability from a realistic combination of risk factors
    risk = (
        (contract == "Month-to-month") * 1.4
        + (tenure < 12) * 1.1
        + (internet_service == "Fiber optic") * 0.5
        + (tech_support == "No") * 0.6
        + (payment_method == "Electronic check") * 0.4
        + (monthly_charges > 80) * 0.5
        - (tenure / 72) * 1.5                      # long-tenure customers churn less
    )
    churn_prob = 1 / (1 + np.exp(-(risk - 1.5)))    # squashes risk score into a 0-1 probability
    churn = rng.binomial(1, churn_prob)             # simulate the actual churn outcome

    df = pd.DataFrame({
        "tenure": tenure,
        "MonthlyCharges": monthly_charges,
        "TotalCharges": total_charges,
        "Contract": contract,
        "InternetService": internet_service,
        "TechSupport": tech_support,
        "PaymentMethod": payment_method,
        "Churn": churn,
    })
    return df


df = generate_synthetic_data()
print("Dataset shape:", df.shape)
df.head()

## 3. Quick EDA (Exploratory Data Analysis)

In [ ]:
print("Overall churn rate:", df["Churn"].mean().round(3))
df.groupby("Contract")["Churn"].mean().round(3)

In [ ]:
# Visualize churn rate by contract type
df.groupby("Contract")["Churn"].mean().sort_values().plot(
    kind="barh", title="Churn Rate by Contract Type", color="salmon"
)
plt.xlabel("Churn Rate")
plt.tight_layout()
plt.show()

## 4. Feature engineering

Creates new, more useful columns and converts text categories into numbers
(models can only work with numbers, not words).

In [ ]:
df["TenureBucket"] = pd.cut(                             # groups tenure into readable bands
    df["tenure"], bins=[-1, 12, 24, 48, 72],
    labels=["0-12mo", "13-24mo", "25-48mo", "49-72mo"]
)

df["AvgMonthlySpendRatio"] = (                            # spend-per-tenure-month, spikes flag risk
    df["TotalCharges"] / (df["tenure"] + 1)
) / df["MonthlyCharges"]

df["ContractRiskScore"] = df["Contract"].map(              # numeric risk weight per contract type
    {"Month-to-month": 2, "One year": 1, "Two year": 0}
)

# Encode remaining categorical text columns as integers so models can use them
cat_cols = ["Contract", "InternetService", "TechSupport", "PaymentMethod", "TenureBucket"]
encoders = {}
for col in cat_cols:
    le = LabelEncoder()                                    # fits a text<->integer mapping
    df[col] = le.fit_transform(df[col])                    # e.g. "Yes"/"No" -> 1/0
    encoders[col] = le                                      # keep the encoder for the dashboard later

df.head()

## 5. Train / test split

In [ ]:
X = df.drop(columns=["Churn"])                              # features
y = df["Churn"]                                              # target label

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y         # stratify keeps churn ratio balanced in both sets
)
print("Train size:", X_train.shape, "  Test size:", X_test.shape)

## 6. Handle class imbalance with SMOTE

Applied to the **training data only** — never touch the test set with SMOTE,
that would leak information and make the evaluation dishonest.

In [ ]:
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)  # synthesizes extra churned examples
print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_bal.value_counts().to_dict())

## 7. Train & compare multiple classifiers

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        eval_metric="logloss", random_state=42
    ),
}

results = {}
for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)                      # train on the SMOTE-balanced data
    preds = model.predict(X_test)                             # class predictions (0/1)
    probs = model.predict_proba(X_test)[:, 1]                 # probability of churn (for ROC-AUC)

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    results[name] = {"accuracy": acc, "roc_auc": auc, "model": model}
    print(f"{name}: accuracy={acc:.3f}  roc_auc={auc:.3f}")

In [ ]:
# Visualize model comparison
comparison_df = pd.DataFrame({k: {"Accuracy": v["accuracy"], "ROC-AUC": v["roc_auc"]} for k, v in results.items()}).T
comparison_df.plot(kind="bar", figsize=(7, 4), title="Model Comparison")
plt.ylabel("Score")
plt.ylim(0.5, 1.0)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Pick the winning model & full report

In [ ]:
best_name = max(results, key=lambda k: results[k]["roc_auc"])  # highest ROC-AUC wins
best_model = results[best_name]["model"]
print(f"Best model: {best_name}\n")
print(classification_report(y_test, best_model.predict(X_test)))

## 9. SHAP explainability

Shows which features drive each customer's churn risk — not just *that* a
customer is high-risk, but *why*.

In [ ]:
explainer = shap.TreeExplainer(best_model)                    # tree-based explainer (fast for XGBoost/RF)
shap_values = explainer.shap_values(X_test.iloc[:200])         # explain a sample of 200 test customers

# Global feature importance (mean absolute SHAP value per feature)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance = pd.Series(mean_abs_shap, index=X_test.columns).sort_values(ascending=False)
print("Top churn drivers (SHAP):")
importance.head(5)

In [ ]:
# Visual SHAP summary plot (bar chart of feature importance)
shap.summary_plot(shap_values, X_test.iloc[:200], plot_type="bar")

In [ ]:
# Optional: full "beeswarm" summary plot — shows direction of impact too
shap.summary_plot(shap_values, X_test.iloc[:200])

## 10. Save model + encoders

These files are loaded by the Streamlit dashboard (`churn_dashboard_app.py`).

In [ ]:
joblib.dump(best_model, "churn_model.pkl")                     # serialize the trained model
joblib.dump(encoders, "encoders.pkl")                           # serialize the label encoders
joblib.dump(list(X.columns), "feature_columns.pkl")             # remember exact column order
print("Saved churn_model.pkl, encoders.pkl, feature_columns.pkl")

## Next steps

- Run `streamlit run churn_dashboard_app.py` in a terminal to launch the live dashboard using the model you just trained.
- Swap the synthetic data in cell 2 for a real customer CSV to train on actual data.